# Lending Club loan selection: one-click rerun

**Capstone, AI for Finance (Idea C).** This notebook reruns the project from the raw data.
In Colab choose **Runtime > Run all**. Each step prints what it did.

Right now it covers **roadmap steps 5 (clean dataset) and 6 (logistic regression + platform benchmark)**. Later steps will be added below as they are approved.

What you need:
* the raw file `accepted_2007_to_2018Q4.csv.gz` (Kaggle `wordsforthewise/lending-club`, about 375 MB), either through a Kaggle API key **or** a copy in your Google Drive;
* nothing else. The code checks the file's SHA-256 fingerprint and stops if it is not the exact file used in the project.

## 1. Get the code
If the repository is private, add a Colab secret called `GITHUB_TOKEN` (key icon on the left) with read access. If it is public, nothing is needed.

In [ ]:
import os, subprocess
REPO = "lucplaisier-coder/lucplaisier"
BRANCH = "main"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists("lucplaisier"):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, "lucplaisier"], check=True)
%cd lucplaisier
!git log -1 --format='Code version: %h  %ad' --date=short

## 2. Install the pinned package versions
The exact versions are in `requirements.txt`. Each step runs as a separate Python process, so Colab's own preinstalled versions do not interfere.

In [ ]:
!pip install -q -r requirements.txt

## 3. Get the raw data (choose ONE option)
* **Option A, Google Drive:** put the file anywhere in your Drive and set `DRIVE_PATH` below.
* **Option B, Kaggle API:** add Colab secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` (from kaggle.com > Settings > API), and leave `DRIVE_PATH` empty.

In [ ]:
DRIVE_PATH = "/content/drive/MyDrive/accepted_2007_to_2018Q4.csv.gz"   # set to "" to use Kaggle

os.makedirs("data/raw", exist_ok=True)
target = "data/raw/accepted_2007_to_2018Q4.csv.gz"
if not os.path.exists(target):
    if DRIVE_PATH:
        from google.colab import drive
        drive.mount("/content/drive")
        !cp "{DRIVE_PATH}" "{target}"
    else:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        !python -c "from src.raw_data import fetch_raw_file; fetch_raw_file('kaggle')"
!ls -l data/raw/

## 4. Step 5: build the clean dataset (about 1 to 3 minutes)
Checks the fingerprint, applies the sample filters, the approved 148-loan rate fix and the fixed input rules, and writes the clean files.
**2014 and 2015 outcomes are written to a sealed file and never summarised** until the methodology freeze (step 8).

In [ ]:
!python -m src.step5_build_clean_dataset

## 5. Look at the step-5 summary tables

In [ ]:
import pandas as pd, json
pd.set_option("display.width", 200)
for name in ["funnel_by_vintage", "resolution_by_vintage", "sample_by_vintage",
             "input_missing_share_by_vintage", "cash_identity_check_2010_2013"]:
    print(f"\n=== {name} ===")
    display(pd.read_csv(f"outputs/step5/{name}.csv"))
summary = json.load(open("outputs/step5/step5_run_summary.json"))
print("Raw file unchanged:", summary["raw_unchanged"])
print("Features frozen:", summary["features_frozen"], "| anything fitted:", summary["anything_fitted"])
print("Sealed outcomes:", summary["sealed_outcomes"])

## 6. Step 6: logistic regression + platform benchmark (a few seconds)
Fits the logit on 2010-2012 only, chooses C on 2013, runs the leakage alarm, and compares fund-all, the platform rule and the logit on 2013 at 80% within each grade. 2014-2015 stay sealed.

In [ ]:
!python -m src.step6_baselines

In [ ]:
for name in ["leakage_alarm_single_input_auc_2013", "logit_tuning_C_2013",
             "predictive_validation_2013", "financial_validation_2013_q80"]:
    print(f"\n=== {name} ===")
    display(pd.read_csv(f"outputs/step6/{name}.csv"))
from IPython.display import Image
display(Image("outputs/step6/reliability_logit_2013.png"))